In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pinn import PINN, SIREN, pde_loss, bc_loss, u_exact

In [2]:
def sample_interior(N):
    pts = torch.rand(N, 2)
    pts.requires_grad_(True)
    return pts


def sample_boundary(N):
    n = N // 4
    # bottom: y=0, x in [0,1]
    bottom = torch.stack([torch.rand(n), torch.zeros(n)], dim=1)
    # top: y=1
    top = torch.stack([torch.rand(n), torch.ones(n)], dim=1)
    # left: x=0
    left = torch.stack([torch.zeros(n), torch.rand(n)], dim=1)
    # right: x=1
    right = torch.stack([torch.ones(n), torch.rand(n)], dim=1)
    return torch.cat([bottom, top, left, right], dim=0)

In [3]:
xy_int = sample_interior(100)
xy_bc = sample_boundary(100)
assert xy_int.shape == (100, 2) and xy_int.requires_grad
assert xy_bc.shape == (100, 2)
# All boundary points should be on edge
assert ((xy_bc[:, 0] == 0) | (xy_bc[:, 0] == 1) |
        (xy_bc[:, 1] == 0) | (xy_bc[:, 1] == 1)).all()
print("Семплирование: ОК")

Семплирование: ОК


In [4]:
torch.manual_seed(42)

model_pinn = PINN(hidden_layers=4, hidden_width=64)
model_siren = SIREN(hidden_layers=4, hidden_width=64, omega_0=30)

optimizer_pinn = torch.optim.Adam(model_pinn.parameters(), lr=1e-3)
optimizer_siren = torch.optim.Adam(model_siren.parameters(), lr=1e-4)

n_params_pinn = sum(p.numel() for p in model_pinn.parameters())
n_params_siren = sum(p.numel() for p in model_siren.parameters())
print(f"PINN параметров: {n_params_pinn}")
print(f"SIREN параметров: {n_params_siren}")

PINN параметров: 12737
SIREN параметров: 12737
